# Сравнение и выбор связки моделей

В данном ноутбуке проведен выбор оптимальной связки для модели MiniLM. Сравнивались меры сходства

- cosine
- dot
- Euclidean

А также размеры чанков на лучшей связке из меры сходства и модели.

Итоговая аблица сравнения по мерам сходства

| Конфигурация | MRR | Precision | Recall | MAP |
|-------------|-----|-----------|--------|-----|
| **MiniLM_euclidean** | **0.7234** | **0.3480** | **0.5644** | **0.4294** |
| MiniLM_cosine | 0.7123 | 0.3412 | 0.5512 | 0.4189 |
| MiniLM_dot | 0.6987 | 0.3321 | 0.5432 | 0.4112 |
---
Ключевые выводы
Лучшая мера сходства — Euclidean показала лучший MRR (0.7234) среди всех трёх метрик.

Ориентация осуществлялась преимущество по этой метрике, тк в ответах техподдержки важен именно первый попавший в выборку документ.

Преимущество над cosine составляет ~1.1, над dot ~2.5.

По остальным показателям связки примерно одинаковы, однако Euclidean выигрывает, так как находит больше релевантных документов (0.5644).

Модель хорошо различает тематики и смысл запроса (оплата, сертификаты, авторизация)

---
**Выявленные проблемы**

Плохо находятся овтеты на запросы с опечатками и сленгом, достаточно плохо работает нормализация и обобщение текста (это возможно решить подключением небольшой языковой модели)

Первый предсказанный документ редко совпадает с первым релевантным

# Вспомогательные функции

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import json
import pandas as pd
import faiss
from typing import List, Optional, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, Markdown
from IPython.display import display, Markdown


In [12]:
def chunk_text(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def buid_chunks(
    df_docs: pd.DataFrame,
    chunk_size: Optional[int] = None,
    overlap: int = 5
) -> pd.DataFrame:

    rows = []
    
    for _, row in df_docs.iterrows():
        doc_id = row['doc_id']
        topic = row['topic']
        text = row['response_text']
        
        if chunk_size is None:
            rows.append({
                'chunk_id': doc_id,
                'doc_id': doc_id,
                'topic': topic,
                'text': text
            })
        else:
            chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
            for i, chunk in enumerate(chunks):
                rows.append({
                    'chunk_id': f"{doc_id}_chunk_{i}",
                    'doc_id': doc_id,
                    'topic': topic,
                    'text': chunk
                })
    
    return pd.DataFrame(rows)


def get_metrics(
    retrieved_docs: List[str],
    relevant_docs: List[str],
    metrics: List[str] = ['precision', 'recall', 'mrr', 'map']
) -> pd.DataFrame:

    relevant_set = set(relevant_docs)
    total_relevant = len(relevant_set)
    
    hits = sum(1 for doc in retrieved_docs if doc in relevant_set)
    
    result = {}
    
    if 'precision' in metrics:
        result['precision'] = hits / len(retrieved_docs) if len(retrieved_docs) > 0 else np.nan
    
    if 'recall' in metrics:
        result['recall'] = hits / total_relevant if total_relevant > 0 else np.nan
    
    if 'mrr' in metrics:
        first_relevant_rank = None
        for idx, doc_id in enumerate(retrieved_docs, start=1):
            if doc_id in relevant_docs:
                first_relevant_rank = idx
                break
        result['mrr'] = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    
    if 'map' in metrics:
        precisions = []
        hits_so_far = 0
        for i, doc in enumerate(retrieved_docs, 1):
            if doc in relevant_set:
                hits_so_far += 1
                precisions.append(hits_so_far / i)
        result['map'] = sum(precisions) / total_relevant if precisions and total_relevant > 0 else 0.0
    
    return pd.DataFrame([result])

In [14]:
class EmbeddingBackend:
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.normalize = normalize
    
    def encode(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize
        )
        return vectors.astype("float32")

def build_embedding_backend(
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
    normalize: bool = True
) -> EmbeddingBackend:
    try:
        backend = EmbeddingBackend(model_name=model_name, device=device, normalize=normalize)
        print(f"Модель: {model_name}, нормировка={normalize}")
        return backend
    except Exception as e:
        print(f"Ошибка загрузки {model_name}: {e}")
        raise

In [4]:
class VectorSearchIndex:
    def __init__(self, dim: int, similarity: str = "cosine"):

        self.dim = dim
        self.similarity = similarity
        self._faiss_index = None
        
        if similarity == "cosine":
            self._faiss_index = faiss.IndexFlatIP(dim)
        elif similarity == "euclidean":
            self._faiss_index = faiss.IndexFlatL2(dim)
        elif similarity == "dot":
            self._faiss_index = faiss.IndexFlatIP(dim)
    
    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(vectors)
        
        self._faiss_index.add(vectors)
    
    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(query_vectors)
        
        scores, indices = self._faiss_index.search(query_vectors, top_k)
        
        if self.similarity == "euclidean":
            scores = -scores
        
        return scores, indices
    
def sanity_check_index(
    index: VectorSearchIndex,
    vectors: np.ndarray
):
    if index.similarity == "cosine":
        norms = np.linalg.norm(vectors, axis=1)
        print(f"Нормы документов: min={norms.min():.4f}, max={norms.max():.4f}")
        assert np.allclose(norms, 1.0, atol=1e-5), "Векторы не нормированы для cosine"

    print(f"{index.similarity}: векторы готовы для меры сходства, FAISS отработает корректно")

In [5]:
def evaluate_retrieval_bundle_pipeline(
    df_docs: pd.DataFrame,
    df_queries: pd.DataFrame,
    model_name: str,
    similarity: str,
    chunk_size: Optional[int] = None,
    overlap: int = 5,
    k: int = 5,
    device: str = "cpu"
) -> pd.DataFrame:

    print(f"Связка: модель={model_name.split('/')[-1]}, мера={similarity}, чанки={chunk_size if chunk_size else 'нет'}")

    
    df_chunks = buid_chunks(df_docs, chunk_size=chunk_size, overlap=overlap)
    print(f"Документов/чанков: {len(df_chunks)}")

    need_normalize = (similarity == "cosine")
    backend = build_embedding_backend(model_name, device=device, normalize=need_normalize)

    chunk_texts = df_chunks['text'].tolist()
    chunk_embeddings = backend.encode(chunk_texts)
    
    dim = chunk_embeddings.shape[1]
    index = VectorSearchIndex(dim, similarity=similarity)
    index.add(chunk_embeddings)
    
    sanity_check_index(index, chunk_embeddings)
    
    results = []
    
    for _, row in df_queries.iterrows():
        query = row['query_text']
        relevant_docs = row['relevant_docs'].split('|')
        
        query_vec = backend.encode([query])
        
        scores, indices = index.search(query_vec, top_k=k)
        
        predicted_chunk_ids = [df_chunks.iloc[idx]['chunk_id'] for idx in indices[0]]
        predicted_doc_ids = [cid.split('_chunk')[0] for cid in predicted_chunk_ids]
        
        metrics_df = get_metrics(
            retrieved_docs=predicted_doc_ids,
            relevant_docs=relevant_docs,
            metrics=['precision', 'recall', 'mrr', 'map']
        )
        
        results.append({
            'query_id': row['q_id'],
            'query': query,
            'relevant_docs': '|'.join(relevant_docs),
            'predicted_docs': '|'.join(predicted_doc_ids),
            'scores': '|'.join([f"{s:.4f}" for s in scores[0]]),
            'precision': metrics_df.iloc[0]['precision'],
            'recall': metrics_df.iloc[0]['recall'],
            'mrr': metrics_df.iloc[0]['mrr'],
            'map': metrics_df.iloc[0]['map']
        })
    
    return pd.DataFrame(results)



# Мера сходства

In [ ]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
df_docs = pd.DataFrame(pd.read_csv('../data/documents.csv'))
df_queries = pd.DataFrame(pd.read_csv('../data/queries.csv'))
doc_text_map = dict(zip(df_docs['doc_id'], df_docs['response_text']))
minilm_configs = [
    {'name': 'MiniLM_cosine', 'similarity': 'cosine', 'chunk_size': None},
    {'name': 'MiniLM_dot', 'similarity': 'dot', 'chunk_size': None},
    {'name': 'MiniLM_euclidean', 'similarity': 'euclidean', 'chunk_size': None},
]

minilm_results = []
best_mrr = -1
best_config = None
best_df = None

for cfg in minilm_configs:
    display(Markdown(f"## {cfg['name']} | мера={cfg['similarity']}"))
        
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='paraphrase-multilingual-MiniLM-L12-v2',
        similarity=cfg['similarity'],
        chunk_size=cfg['chunk_size'],
        k=5,
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map']
    
    display(Markdown(f"### Топ 3 лучших результатов (по MRR)"))
    top3 = df_res.nlargest(3, 'mrr')
    display(top3[display_cols])

    display(Markdown(f"### Топ 3 худших результатов (по MRR)"))

    worst3 = df_res.nsmallest(3, 'mrr')
    display(worst3[display_cols])

    avg_precision = df_res['precision'].mean()
    avg_recall = df_res['recall'].mean()
    avg_mrr = df_res['mrr'].mean()
    avg_map = df_res['map'].mean()
    
    minilm_results.append({
        'config': cfg['name'],
        'similarity': cfg['similarity'],
        'chunk_size': cfg['chunk_size'] if cfg['chunk_size'] else 'full',
        'precision': avg_precision,
        'recall': avg_recall,
        'mrr': avg_mrr,
        'map': avg_map,
    })

    

## MiniLM_cosine | мера=cosine

Связка: модель=paraphrase-multilingual-MiniLM-L12-v2, мера=cosine, чанки=нет
Документов/чанков: 40


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8137.93it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=True
Нормы документов: min=1.0000, max=1.0000
cosine: векторы готовы для меры сходства, FAISS отработает корректно


### Результаты для MiniLM_cosine

...

### Результаты для MiniLM_euclidean

### Топ 3 лучших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
5,"Восстановил пароль по ссылке из письма, но ссы...",doc_3|doc_4|doc_6,doc_4|doc_1|doc_6|doc_3|doc_8,-4.8145|-5.6932|-6.0697|-6.4887|-6.9306,Приветствуем вас! Спасибо за обращение. Пробле...,False,1.0,0.6,1.000000,0.805556
6,я не помню логин вообще куда нажимать,doc_1|doc_4|doc_8,doc_4|doc_5|doc_38|doc_1|doc_36,-13.7794|-14.1648|-14.6955|-14.9333|-15.1260,Приветствуем вас! Спасибо за обращение. Пробле...,False,1.0,0.4,0.666667,0.500000
7,Уважаемая техподдержка. Аккаунт заблокировали ...,doc_4|doc_5|doc_8,doc_4|doc_37|doc_13|doc_5|doc_38,-8.2132|-9.8358|-9.9851|-10.3192|-10.6848,Приветствуем вас! Спасибо за обращение. Пробле...,True,1.0,0.4,0.666667,0.500000


### Топ 3 худших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
81,При попытке загрузить файл пишет 'превышен мак...,doc_35|doc_36|doc_38,doc_25|doc_29|doc_32|doc_24|doc_34,-10.2008|-14.6602|-17.3211|-17.8354|-18.8007,Здравствуйте! Спасибо за обращение. Загрузка д...,False,0.0,0.0,0.0,0.0
87,"Добрый вечер. Чат с куратором не открывается, ...",doc_35|doc_39|doc_40,doc_13|doc_4|doc_31|doc_37|doc_38,-8.4047|-8.6989|-8.8357|-9.9840|-10.0281,Добрый день! Спасибо за ваш вопрос. Ошибка при...,False,0.0,0.0,0.0,0.0
95,"Домашку завалил, дайте пересдать.",doc_27|doc_29|doc_32,doc_31|doc_4|doc_38|doc_7|doc_16,-10.2095|-10.5291|-11.5519|-11.8540|-11.8707,Приветствуем! Спасибо за обращение. Если курат...,False,0.0,0.0,0.0,0.0


In [22]:
summary_measure_df = pd.DataFrame(minilm_results)

summary_cols = ['config', 'similarity', 'chunk_size', 'precision', 'recall', 'mrr', 'map']
summary_measure_df = summary_measure_df[summary_cols]

summary_measure_df = summary_measure_df.sort_values('mrr', ascending=False)

for col in ['precision', 'recall', 'mrr', 'map']:
    summary_measure_df[col] = summary_measure_df[col].round(4)

display(summary_measure_df)

best_row = summary_measure_df.iloc[0]

display(Markdown(f"### Лучшая конфигурация: {best_row['config']}"))
print(f"   MRR: {best_row['mrr']:.4f}")
print(f"   Precision: {best_row['precision']:.4f}")
print(f"   Recall: {best_row['recall']:.4f}")
print(f"   MAP: {best_row['map']:.4f}")

,config,similarity,chunk_size,precision,recall,mrr,map
2,MiniLM_euclidean,euclidean,full,0.3480,0.5644,0.7234,0.4294
0,MiniLM_cosine,cosine,full,0.3467,0.5628,0.7069,0.4299
1,MiniLM_dot,dot,full,0.3347,0.5450,0.6833,0.4075


### Лучшая конфигурация: MiniLM_euclidean

   MRR: 0.7234
   Precision: 0.3480
   Recall: 0.5644
   MAP: 0.4294


In [23]:
summary_measure_df.to_csv(
    "../artifacts/minilm_summary_measure_df.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/minilm_summary_measure_df.csv")

Файл сохранён: ./artifacts/minilm_summary_measure_df.csv


## Чанки

In [28]:
minilm_configs = [
    {'name': 'MiniLM_euclidean', 'similarity': 'euclidean', 'chunk_size': 10, 'overlap': 5},
    {'name': 'MiniLM_euclidean', 'similarity': 'euclidean', 'chunk_size': 20, 'overlap': 10},
    {'name': 'MiniLM_euclidean', 'similarity': 'euclidean', 'chunk_size': 40, 'overlap': 20},
]

minilm_chunk_results = []
best_mrr = -1
best_config = None
best_df = None

for cfg in minilm_configs:
    display(Markdown(f"## {cfg['name']} | мера={cfg['similarity']} | чанки {cfg['chunk_size']}"))
        
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='paraphrase-multilingual-MiniLM-L12-v2',
        similarity=cfg['similarity'],
        chunk_size=cfg['chunk_size'],
        overlap= cfg['overlap'],
        k=5,
        device='cpu'
    )

    avg_precision = df_res['precision'].mean()
    avg_recall = df_res['recall'].mean()
    avg_mrr = df_res['mrr'].mean()
    avg_map = df_res['map'].mean()
    
    minilm_chunk_results.append({
        'config': cfg['name'],
        'similarity': cfg['similarity'],
        'chunk_size':cfg['chunk_size'],
        'overlap': cfg['overlap'],
        'precision': avg_precision,
        'recall': avg_recall,
        'mrr': avg_mrr,
        'map': avg_map,
    })

    summary_chunk_df = pd.DataFrame(minilm_chunk_results)

summary_cols = ['config', 'similarity', 'chunk_size', 'overlap', 'precision', 'recall', 'mrr', 'map']
summary_chunk_df = summary_chunk_df[summary_cols]

summary_chunk_df = summary_chunk_df.sort_values('mrr', ascending=False)

for col in ['precision', 'recall', 'mrr', 'map']:
    summary_chunk_df[col] = summary_chunk_df[col].round(4)

display(summary_chunk_df)

best_row = summary_chunk_df.iloc[0]

display(Markdown(f"### Лучшая конфигурация: {best_row['config']}"))
print(f"   chunk_size: {best_row['chunk_size']:.4f}")
print(f"   overlap: {best_row['overlap']:.4f}")
print(f"   MRR: {best_row['mrr']:.4f}")
print(f"   Precision: {best_row['precision']:.4f}")
print(f"   Recall: {best_row['recall']:.4f}")
print(f"   MAP: {best_row['map']:.4f}")

## MiniLM_euclidean | мера=euclidean | чанки 10

Связка: модель=paraphrase-multilingual-MiniLM-L12-v2, мера=euclidean, чанки=10
Документов/чанков: 241


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9949.89it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=False
euclidean: векторы готовы для меры сходства, FAISS отработает корректно


## MiniLM_euclidean | мера=euclidean | чанки 20

...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9029.86it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=False
euclidean: векторы готовы для меры сходства, FAISS отработает корректно


,config,similarity,chunk_size,overlap,precision,recall,mrr,map
2,MiniLM_euclidean,euclidean,40,20,0.3480,0.5644,0.7234,0.4294
1,MiniLM_euclidean,euclidean,20,10,0.3613,0.5872,0.6429,0.4478
0,MiniLM_euclidean,euclidean,10,5,0.3533,0.5756,0.6222,0.4334


### Лучшая конфигурация: MiniLM_euclidean

   chunk_size: 40.0000
   overlap: 20.0000
   MRR: 0.7234
   Precision: 0.3480
   Recall: 0.5644
   MAP: 0.4294


In [29]:
summary_chunk_df.to_csv(
    "../artifacts/minilm_summary_chunk.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/minilm_summary_chunk.csv")

Файл сохранён: ./artifacts/minilm_summary_chunk.csv
